## Check out the trace:

https://platform.openai.com/traces

In [15]:
import asyncio
from agents import Runner, trace, gen_trace_id, Agent, WebSearchTool
from typing import List, Dict
from dotenv import load_dotenv
from typing import List, Optional
from pydantic import BaseModel, Field

load_dotenv(override=True)

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 

# Agent prompt: fetch official list from USCIS
QUESTION_INSTRUCTIONS = """
1. 🧭 **Determine the active civics test version & metadata**
   - Search USCIS.gov (e.g., policy manual, “Check for Test Updates” page) to confirm which version (e.g., “2008 version”) is currently used during naturalization interviews and capture its metadata:
     - version name (e.g., “2008 version”)
     - USCIS last reviewed/updated date (e.g., “01/21/2025”)

2. 📄 **Extract the official question list**
   - Based on that version:
     - For 2008: fetch “100 Civics Questions and Answers” PDF/text.
     - For any future version: fetch that full official set.
   - Extract exactly *n* questions (n = number in that version), preserving official numbering and exact wording.

3. 🧾 **Return JSON only**, matching the Pydantic schema:
   ```json
   {
     "civic_test_version": "<e.g. \"2008 version\">",
     "uscis_last_updated": "<MM/DD/YYYY>",
     "questions": [
       { "id": 1, "question": "What is the supreme law of the land?" },
       … exactly n items …
     ]
   }
"""

class CivicsQuestionItem(BaseModel):
    id: int = Field(..., description="Official USCIS question number if available in source material", ge=1, le=100)
    question: str = Field(..., description="A civics question from the USCIS official list")

class CivicsQuestionsList(BaseModel):
    civic_test_version: str = Field(..., description="The version of the civics test (e.g., '2008 version', 'Updated 2024', etc.)")
    uscis_last_updated: str = Field(
        ..., description="Date when USCIS last reviewed/updated that test version (e.g., '01/21/2025')"
    )
    questions: List[CivicsQuestionItem] = Field(
        ..., 
        description="Complete set of 100 USCIS civics questions",
        min_items=100,
        max_items=100
    )

async def fetch_civics_questions():
    with trace("GET_Civics_Questions"):
        print("🌟🌟🌟🌟 START --> fetch_questions 🌟🌟🌟🌟")
        civic_question_getter_agent = Agent(
            name="QuestionGetterAgent",
            instructions=QUESTION_INSTRUCTIONS,
            model="gpt-4o-mini",
            tools=[WebSearchTool(search_context_size="medium")],
            output_type=CivicsQuestionsList,
        )
        result = await Runner.run(civic_question_getter_agent, input="List the last update 100 civics questions.")
        print(result.final_output)
        print("🌟🌟🌟🌟 END --> fetch_questions 🌟🌟🌟🌟")
        return result.final_output  # list of questions
    
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 

## STEP @ ANSWERS AGENTS 
### ANSWERS
ANSWER_INSTRUCTIONS = """
You are the civics AnswerFinderAgent. For each civics question:

1. 🔍 **Use only the official USCIS resources** to find the answer. Do not generate answers independently.
2. 📄 Refer to the following official documents:
   - [100 Civics Questions and Answers (2008 version)](https://www.uscis.gov/sites/default/files/document/questions-and-answers/100q.pdf)
   - [Study for the Test](https://www.uscis.gov/citizenship/find-study-materials-and-resources/study-for-the-test)
3. 🏛 If the question is state-specific (e.g., Senators, Representative, Governor, capital), use the provided state value to ensure current information. Handle DC or U.S. territories correctly.
4. 📋 Return **all acceptable answer variants** as listed in the official resources (including official names, spellings, nicknames, current incumbents, etc.).

⚠️ **Important**: The output **must** be valid JSON that fully matches the `CivicsQuestionWithAnswersItem` schema:
```json
{
  "id": <int 1–100>,
  "question": "<exact question wording>",
  "answers": ["<first acceptable answer>", "<second variant>", ...]
}
"""

class CivicsQuestionWithAnswersItem(BaseModel):
    """
    Represents a single USCIS civics test question along with its acceptable answers.

    Fields:
    - id (int): Official USCIS question number (1–100)
    - question (str): Exact wording of the civics question
    - answers (List[str]): One or more valid answer strings (including current names, variations, etc.)
    """
    id: int = Field(..., ge=1, le=100, description="Official question number (1–100)")
    question: str = Field(..., description="Exact civics question wording")
    answers: List[str] = Field(..., description="List of acceptable answers")


async def fetch_civics_answers(qs: CivicsQuestionsList, state: str):
    print("🌟🌟🌟🌟 START --> fetch_civics_answers 🌟🌟🌟🌟")
    agent = Agent(
        name="AnswerFinderAgent",
        instructions=ANSWER_INSTRUCTIONS,
        model="gpt-4o-mini",
        tools=[WebSearchTool(search_context_size="medium")],
        output_type=CivicsQuestionWithAnswersItem,
    )
    
    async def task(item):
        prompt = (
            f"Question ID: {item.id}\n"
            f"Question: {item.question}\n"
            f"State: {state}\n"
            f"Civic Test Version: {qs.civic_test_version}\n\n"
            f"Civic Test Last Updated: {qs.uscis_last_updated}\n\n"
            "Please provide all acceptable answers using web search."
        )
        with trace(f"Answer_Finder_{item.id}"):
            result = await Runner.run(agent, input=prompt)
        return result.final_output
    

    items = await asyncio.gather(*(task(item) for item in qs.questions))
    print(items)

    print("🌟🌟🌟🌟 END --> fetch_civics_answers 🌟🌟🌟🌟")

    return items

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 


async def runCivicsQuestionWorkflow(state=None):
    print("🏃🏻‍♀️🏃🏻‍♀️🏃🏻‍♀️🏃🏻‍♀️🏃🏻‍♀️  START --> runCivicsQuestionWorkflow 🏃🏻‍♀️🏃🏻‍♀️🏃🏻‍♀️🏃🏻‍♀️🏃🏻‍♀️")
    print("🔍 questions from fetch_civics_questions() --> ")
    questions = await fetch_civics_questions()
    print(questions)

    print("🔍 answers from fetch_civics_answers() --> ")
    questions_answers_list = await fetch_civics_answers(questions, state)
    print(questions_answers_list)

    print("🏃🏻‍♀️🏃🏻‍♀️🏃🏻‍♀️🏃🏻‍♀️🏃🏻‍♀️ END --> runCivicsQuestionWorkflow 🏃🏻‍♀️🏃🏻‍♀️🏃🏻‍♀️🏃🏻‍♀️🏃🏻‍♀️")


await runCivicsQuestionWorkflow("Oklahoma")

🏃🏻‍♀️🏃🏻‍♀️🏃🏻‍♀️🏃🏻‍♀️🏃🏻‍♀️  START --> runCivicsQuestionWorkflow 🏃🏻‍♀️🏃🏻‍♀️🏃🏻‍♀️🏃🏻‍♀️🏃🏻‍♀️
🔍 questions from fetch_civics_questions() --> 
🌟🌟🌟🌟 START --> fetch_questions 🌟🌟🌟🌟


Error getting response: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}. (request_id: req_84185e65ed42bc00cbd01307779d23f7)


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
from agents import Runner, trace, gen_trace_id, Agent, WebSearchTool
from pydantic import BaseModel, Field
from typing import List, Optional
from enum import Enum
from dotenv import load_dotenv

load_dotenv(override=True)

class CivicsTopics(str, Enum):
    """Main topic categories for USCIS civics test"""
    AMERICAN_GOVERNMENT = "AMERICAN GOVERNMENT"
    AMERICAN_HISTORY = "AMERICAN HISTORY"
    INTEGRATED_CIVICS = "INTEGRATED CIVICS"

class AmericanGovernmentSubTopics(str, Enum):
    """Sub-topics under American Government"""
    PRINCIPLES_OF_DEMOCRACY = "A: Principles of American Democracy"
    SYSTEM_OF_GOVERNMENT = "B: System of Government"
    RIGHTS_AND_RESPONSIBILITIES = "C: Rights and Responsibilities"

class AmericanHistorySubTopics(str, Enum):
    """Sub-topics under American History"""
    COLONIAL_PERIOD_AND_INDEPENDENCE = "A: Colonial Period and Independence"
    EIGHTEEN_HUNDREDS = "B: 1800s"
    RECENT_AMERICAN_HISTORY = "C: Recent American History"

class IntegratedCivicsSubTopics(str, Enum):
    """Sub-topics under Integrated Civics"""
    GEOGRAPHY = "A: Geography"
    SYMBOLS = "B: Symbols"
    HOLIDAYS = "C: Holidays"

# Union type for all sub-topics
CivicsSubTopics = AmericanGovernmentSubTopics | AmericanHistorySubTopics | IntegratedCivicsSubTopics


class CivicsQuestionItem(BaseModel):
    question: str = Field(..., description="A civics question from the USCIS official list")
    answers: List[str] = Field(..., description="All acceptable answers, including current updates and variations")
    topic: CivicsTopics = Field(..., description="Main topic category")
    sub_topic: CivicsSubTopics = Field(..., description="Sub-topic category")
    official_number: Optional[int] = Field(None, description="Official USCIS question number if available in source material", ge=1, le=100)

class CivicsQuestions(BaseModel):
    civic_test_version: str = Field(..., description="The version of the civics test (e.g., '2008 version', 'Updated 2024', etc.)")
    questions: List[CivicsQuestionItem] = Field(
        ..., 
        description="Complete set of 100 USCIS civics questions with current answers",
        min_items=100,
        max_items=100
    )


INSTRUCTIONS = """
You are the USCIS civics interviewer agent. Your task is to generate the complete, up-to-date list of 100 civics questions used in the U.S. citizenship test.

REQUIREMENTS:
1. Use web search to find the most current version of USCIS civics test questions
2. Apply ALL current updates for questions with variable answers by searching for latest information
3. Generate exactly 100 questions in the specified JSON format
4. For state-specific questions, use the provided state/territory to find current answers

SEARCH STRATEGY:
1. **Find Current Test Version**: Search for the latest USCIS civics test version and questions
2. **Get Official Question List**: Search for the complete official USCIS civics test questions - focus on question content rather than numbering
3. **Get Topic Structure**: Search for the official topic and sub-topic organization from USCIS materials and map questions by content to their appropriate categories
   - AMERICAN GOVERNMENT: A: Principles of American Democracy, B: System of Government, C: Rights and Responsibilities
   - AMERICAN HISTORY: A: Colonial Period and Independence, B: 1800s, C: Recent American History
   - INTEGRATED CIVICS: A: Geography, B: Symbols, C: Holidays
4. **Find Current Updates**: Search for any recent updates at USCIS website
5. **Get Current Officeholders**: Search for current information on all time-sensitive questions

STATE-SPECIFIC QUESTIONS (use provided state/territory):
When a state is provided, search for current information for these specific questions:
- "Who is one of your state's U.S. Senators now?" 
  * Search for current U.S. Senators for the specified state
  * For DC/territories: "D.C. has no U.S. Senators" or "[Territory] has no U.S. Senators"
- "Name your U.S. Representative."
  * Search for current U.S. Representative for the specified state/district
  * For territories: Include non-voting delegates/commissioners or state "no voting Representatives"
- "Who is the Governor of your state now?"
  * Search for current Governor of the specified state
  * For DC: "D.C. does not have a Governor"
- Any state capital questions: If asked about state capital, search for the capital of the specified state

SEARCH INSTRUCTIONS:
- Always search for the most current information - do not rely on any hardcoded data
- Search multiple sources to verify accuracy
- For time-sensitive questions, prioritize recent information
- Cross-reference official government websites (.gov domains)
- Include all acceptable answer variations as provided by USCIS

OUTPUT FORMAT:
Generate exactly 100 question-answer pairs following the CivicsQuestions schema. Each question must include:
- The exact question text from the official USCIS list (this is the primary identifier)
- All acceptable answers, including variations and alternative phrasings
- Current, verified information for all time-sensitive questions
- The correct topic and sub-topic classification from official USCIS materials
- Official question number if clearly provided in source materials (optional)
- State-specific answers when state is provided

TOPIC STRUCTURE (use these exact enum values):
Map questions to topics based on their content, not position:
- AMERICAN GOVERNMENT
  * A: Principles of American Democracy (democracy, constitution, rule of law questions)
  * B: System of Government (branches, officials, elections questions)
  * C: Rights and Responsibilities (citizenship, responsibilities, rights questions)
- AMERICAN HISTORY
  * A: Colonial Period and Independence (founding, revolution, early history questions)
  * B: 1800s (civil war, expansion, 19th century questions)
  * C: Recent American History (world wars, modern history questions)
- INTEGRATED CIVICS
  * A: Geography (states, rivers, regions questions)
  * B: Symbols (flag, anthem, monuments questions)
  * C: Holidays (national holidays questions)

IMPORTANT: Use the exact enum values defined in the schema. The agent must identify questions by their content and map them to appropriate categories.

QUALITY CHECKS:
- Ensure all 100 questions are included (no duplicates, no omissions)
- Identify questions by their content, not by assumed position numbers
- Verify that all variable answers reflect current information
- Include all acceptable answer variations as provided by USCIS
- Maintain exact question wording from the official source
- Confirm correct topic and sub-topic classification based on question content
- Verify the test version being used matches the official USCIS materials
"""

civics_question_agent = Agent(
    name="CivicsQuestionAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    tools=[WebSearchTool(search_context_size="medium")],
    output_type=CivicsQuestions,
)

async def generate_civics_questions(state: Optional[str] = None):
    """
    Generate the complete USCIS civics questions list with current information
    
    Args:
        state: The state or territory for state-specific questions (e.g., "California", "Texas", "DC")
               If None, will provide generic variable answer guidance
    """
    
    # Build the prompt based on whether state is provided
    if state:
        state_instruction = f"""
        IMPORTANT: The user is from {state}. For state-specific questions, provide current answers for {state}:
        - "Who is one of your state's U.S. Senators now?" → Search for current U.S. Senators from {state}
        - "Name your U.S. Representative." → Search for current U.S. Representative from {state}
        - "Who is the Governor of your state now?" → Search for current Governor of {state}
        - Any state capital questions → Search for the capital of {state}
        
        Special cases:
        - If {state} is DC: DC has no U.S. Senators and no Governor
        - If {state} is a territory: May have non-voting delegates instead of Representatives
        """
    else:
        state_instruction = """
        For state-specific questions, provide the generic USCIS guidance:
        - "Answers will vary" with instructions to visit official websites
        - Include special notes for DC residents and territories
        """
    
    prompt = f"""
    Generate the complete, current list of 100 USCIS civics questions with updated answers.
    
    SEARCH FOR ALL CURRENT INFORMATION - do not use any hardcoded data:
    1. Find the latest version of USCIS civics test questions
    2. Search for current President, Vice President, Speaker of the House, Chief Justice
    3. Search for current national holidays list
    4. Search for any recent USCIS test updates
    
    {state_instruction}
    
    Ensure all answers are current and verified through web search.
    """
    
    with trace("USCIS_Civics_Questions"):
        result = await Runner.run(civics_question_agent, prompt)
    
        print("=== RAW OUTPUT FROM AGENT ===")
        print(result.raw_output)  # Shows what the LLM returned before parsing
        print("=== PARSE ERRORS (if any) ===")
        print(result.validation_errors)  # Show what failed during parsing

        return result.final_output

california_questions = await generate_civics_questions("California")
print(f"Test Version: {california_questions.civic_test_version}")
print(f"Generated {len(california_questions.questions)} questions")


ModelBehaviorError: Invalid JSON when parsing {"civic_test_version":"2008 version","questions":[{"question":"What is the supreme law of the land?","answers":["the Constitution"],"topic":"AMERICAN GOVERNMENT","sub_topic":"A: Principles of American Democracy","official_number":1},{"question":"What does the Constitution do?","answers":["sets up the government","defines the government","protects basic rights of Americans"],"topic":"AMERICAN GOVERNMENT","sub_topic":"A: Principles of American Democracy","official_number":2},{"question":"The idea of self-government is in the first three words of the Constitution. What are these words?","answers":["We the People"],"topic":"AMERICAN GOVERNMENT","sub_topic":"A: Principles of American Democracy","official_number":3},{"question":"What is an amendment?","answers":["a change (to the Constitution)","an addition (to the Constitution)"],"topic":"AMERICAN GOVERNMENT","sub_topic":"A: Principles of American Democracy","official_number":4},{"question":"Wha